# Math 02: Calculus — Derivatives & the Power Rule**Learning Objectives:**1. Understand the derivative as the slope of a function at a point2. Compute derivatives via the difference quotient (limit definition)3. Apply the Power Rule to differentiate polynomials efficiently4. Implement numerical and symbolic differentiation in Python**Prerequisites:** Math 01 (Algebra), basic function concepts**Estimated time:** 120 minutes**Textbook:** *Justin Skycak — Calculus* §1.3–1.4

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "math_02_calculus_derivatives",    "level": 1,    "title": "Math 02: Calculus \u2014 Derivatives & the Power Rule",    "prerequisites": [        "Math 01 (Algebra)",        "basic function concepts"    ],    "skills_taught": [        "lesson.math_02_calculus_derivatives"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "math_03_calculus_chain_rule.ipynb"}SKILLS = [    {        "id": "lesson.math_02_calculus_derivatives",        "title": "Lesson Math 02 Calculus Derivatives",        "notebook": "math_02_calculus_derivatives.ipynb",        "level": 1    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "math_02_calculus_derivatives.ipynb",        "level": 1    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Math 01 (Algebra), basic function concepts.

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.math_02_calculus_derivatives', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setup!pip install -q sympy matplotlib plotly ipywidgetsimport sympy as sp; sp.init_printing()import numpy as np, matplotlib.pyplot as pltprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### What is a derivative?> 📖 *From Calculus §1.3:* The **derivative** of a function is the function's slope at a particular point.We approximate the slope at $x$ by picking a nearby point $x + \Delta x$ and computing:$$\text{slope} \approx \frac{f(x+\Delta x) - f(x)}{\Delta x}$$This is the **difference quotient**. The derivative is the limit as $\Delta x \to 0$:$$f'(x) = \lim_{\Delta x \to 0} \frac{f(x+\Delta x)-f(x)}{\Delta x}$$

### Guided Proof — Derivative of $f(x) = x^2$$$f'(x) = \lim_{\Delta x \to 0}\frac{(x+\Delta x)^2 - x^2}{\Delta x} = \lim_{\Delta x \to 0}\frac{x^2+2x\Delta x+(\Delta x)^2-x^2}{\Delta x}$$$$= \lim_{\Delta x \to 0}\frac{2x\Delta x + (\Delta x)^2}{\Delta x} = \lim_{\Delta x \to 0}(2x + \Delta x) = 2x \quad \square$$So the slope of $x^2$ at any point $x$ is $2x$. At $x=-2$ the slope is $-4$ (falling), at $x=0$ slope is $0$ (flat valley), at $x=3$ slope is $6$ (rising).

In [ ]:
# Symbolic verification with SymPyimport sympy as spx, dx = sp.symbols('x Delta_x')f = x**2diff_quot = ((x+dx)**2 - x**2) / dxprint('Difference quotient:', sp.expand(diff_quot))print('Limit as Δx→0:    ', sp.limit(diff_quot, dx, 0))print('SymPy derivative:  ', sp.diff(f, x))

### The Power Rule> 📖 *From Calculus §1.4:* The **power rule** states that for $f(x) = x^n$:> $$f'(x) = nx^{n-1}$$| $f(x)$ | $f'(x)$ | Rule applied ||---|---|---|| $x^3$ | $3x^2$ | bring 3 down, reduce power || $x^{-1} = 1/x$ | $-x^{-2}$ | works for negative exponents || $x^{1/2} = \sqrt{x}$ | $\frac{1}{2}x^{-1/2}$ | works for fractional exponents || $x^0 = 1$ (constant) | $0$ | constants have zero slope |For a sum of terms: differentiate each term individually.For a constant multiple: $\frac{d}{dx}[cf(x)] = c \cdot f'(x)$.**Example:** $f(x)=3x^4-2x^2+5x-7$$$f'(x) = 12x^3 - 4x + 5$$

### Comprehension Check ✅1. What is the derivative of $f(x) = 7x^3 - x + 4$?2. Why is the derivative of a constant always 0?3. What does a negative derivative tell you about the function?<details><summary>💡 Explanation after a retrieval attempt</summary>1. $f'(x)=21x^2-1$2. A constant has no slope — it's a flat horizontal line.3. The function is *decreasing* at that point.</details>

---## Phase 0 — Math Foundation Practice### Worked Example: Numerical Derivative

In [ ]:
# Computing the derivative numerically vs analyticallyimport numpy as npimport matplotlib.pyplot as pltdef f(x): return x**3 - 2*x + 1def f_prime_exact(x): return 3*x**2 - 2  # by power rule# Numerical derivative using difference quotientdef numerical_derivative(f, x, dx=1e-7):    return (f(x + dx) - f(x)) / dx# Compare at x = 2x_val = 2.0print(f'Exact derivative at x={x_val}:    {f_prime_exact(x_val)}')print(f'Numerical derivative at x={x_val}: {numerical_derivative(f, x_val)}')print(f'Error: {abs(f_prime_exact(x_val) - numerical_derivative(f, x_val)):.2e}')

### 🎯 Your TurnUse the difference quotient to find the derivative of $f(x)=x^4$ by hand.Then verify with SymPy.

In [ ]:
def power_rule_derivative(coefficients):    '''Apply the power rule to a polynomial.    coefficients: list [a_n, a_{n-1}, ..., a_1, a_0] (highest degree first)    Returns: list of derivative coefficients    '''    # YOUR CODE HERE    pass# Test: f(x) = 3x^3 - 2x^2 + x - 5 → f'(x) = 9x^2 - 4x + 1# assert power_rule_derivative([3, -2, 1, -5]) == [9, -4, 1]# print('Phase 0 passed ✅')

<details><summary>💡 Hint</summary>For term $a_k x^k$, the derivative is $k \cdot a_k \cdot x^{k-1}$.Iterate from the highest power down, multiplying each coefficient by its exponent.</details>

---## Phase 1 — Micro-Scaffolded Algorithm Construction### Step 1: Difference Quotient FunctionWrite a general numerical derivative function.

In [ ]:
def derivative(f, x, dx=1e-7):    '''Approximate f'(x) using the difference quotient.'''    # YOUR CODE HERE    pass# Test:# import math# assert abs(derivative(lambda x: x**2, 3) - 6.0) < 1e-5# assert abs(derivative(math.sin, 0) - 1.0) < 1e-5  # cos(0) = 1# print('Step 1 passed ✅')

### Step 2: Central Difference (Better Approximation)The central difference $\frac{f(x+h)-f(x-h)}{2h}$ is more accurate than the forward difference.

In [ ]:
def derivative_central(f, x, dx=1e-7):    '''Approximate f'(x) using the central difference quotient.'''    # YOUR CODE HERE    pass# Compare accuracy:# import math# x0 = 1.0# exact = math.cos(x0)  # derivative of sin# fwd = derivative(math.sin, x0)# ctr = derivative_central(math.sin, x0)# print(f'Exact:   {exact:.15f}')# print(f'Forward: {fwd:.15f}  error={abs(fwd-exact):.2e}')# print(f'Central: {ctr:.15f}  error={abs(ctr-exact):.2e}')

### Step 3: Visualize the Tangent Line (Spaced Repetition — uses slope-intercept from Math 01)

In [ ]:
def plot_tangent(f, f_prime, x0):    '''Plot f(x) and its tangent line at x0.'''    import matplotlib.pyplot as plt, numpy as np    xs = np.linspace(x0 - 3, x0 + 3, 300)    slope = f_prime(x0)    tangent = slope * (xs - x0) + f(x0)    plt.figure(figsize=(8, 5))    plt.plot(xs, f(xs), 'b-', lw=2, label='f(x)')    plt.plot(xs, tangent, 'r--', lw=2, label=f'tangent (slope={slope:.2f})')    plt.plot(x0, f(x0), 'ko', ms=8, zorder=5)    plt.legend(); plt.grid(alpha=0.3); plt.title(f'Tangent Line at x={x0}')    plt.tight_layout(); plt.show()# Example: f(x) = x^3 - 2x + 1, tangent at x = 1plot_tangent(lambda x: x**3-2*x+1, lambda x: 3*x**2-2, 1.0)

---## Phase 2 — Experimental Verification### Pathological Case: Choosing dx Too SmallIf $\Delta x$ is too small, floating-point subtraction in $f(x+\Delta x)-f(x)$ cancels most digits.

In [ ]:
import numpy as np, matplotlib.pyplot as pltdef f(x): return x**3def f_prime(x): return 3*x**2  # exactx0 = 1.0dx_values = np.logspace(-1, -16, 100)errors = [abs((f(x0+dx)-f(x0))/dx - f_prime(x0)) for dx in dx_values]plt.figure(figsize=(9,5))plt.loglog(dx_values, errors, 'r-', lw=2)plt.xlabel('Δx'); plt.ylabel('Absolute Error')plt.title('Error in Numerical Derivative vs Step Size')plt.axvline(x=1e-8, color='g', ls='--', label='Sweet spot ≈ 1e-8')plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()print('Too large Δx → truncation error. Too small Δx → rounding error.')

### 🔍 Reflection1. What step size minimizes error?2. Why does error *increase* when $\Delta x < 10^{-8}$?<details><summary>💡 Explanation after a retrieval attempt</summary>Optimal $\Delta x \approx \sqrt{\epsilon_{\text{machine}}} \approx 10^{-8}$ for float64.Below that, $f(x+\Delta x)$ and $f(x)$ are so close that their difference loses all significant digits.</details>

---## Phase 3 — Olympiad Extension### Challenge: Symbolic DifferentiatorBuild a function that takes a polynomial as a list of coefficients and returnsits derivative as a list — **without using SymPy**. Then extend it to compute the$n$-th derivative.

In [ ]:
def nth_derivative(coeffs, n=1):    '''Compute the n-th derivative of a polynomial.    coeffs: [a_n, ..., a_1, a_0] highest degree first    Returns: coefficient list of the n-th derivative    '''    # YOUR CODE HERE (no scaffolding!)    pass# Test: f(x) = x^4, f''(x) = 12x^2 → [12, 0, 0]# assert nth_derivative([1,0,0,0,0], 2) == [12, 0, 0]

### Chalkboard Defense1. **Correctness:** Prove that the central difference has error $O(h^2)$ vs $O(h)$ for forward difference.2. **Complexity:** What is the cost of computing the $n$-th derivative of a degree-$d$ polynomial?3. **Stability:** Explain the U-shaped error curve from Phase 2.*(Write your defense below)*<details><summary>💡 Defense sketch</summary>Taylor expand: $f(x+h) = f(x)+hf'(x)+\frac{h^2}{2}f''(x)+O(h^3)$.Forward difference: error term is $\frac{h}{2}f''(x)$ → $O(h)$.Central: $f(x+h)-f(x-h)=2hf'(x)+O(h^3)$ → divide by $2h$ gives error $O(h^2)$.</details>

---### 📚 Further Reading- *Calculus* §1.5–1.6: Chain Rule, Product/Quotient Rule- **Next:** Math 03 (Chain Rule)

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.math_02_calculus_derivatives', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='math_03_calculus_chain_rule.ipynb')